# Natural Presenter AI — F5-TTS Per-Speaker Fine-Tuning (Google Colab Pro — L4 / A100)

This notebook fine-tunes **`F5TTS_v1_Base`** on a single speaker's verified dataset (e.g., `vn_01` Binh or `vn_02` Ege) while **preserving authentic non-native English accents, breaths, and `uh`/`um`/`eh` disfluencies**.

### Before You Run
1. In Colab menu: **Runtime → Change runtime type → Hardware accelerator: `L4 GPU`** (23GB VRAM — or `A100` if preferred).
2. Upload your zipped dataset (e.g., `vn_01_f5_dataset.zip`, ~61 MB) into your Google Drive at:
   `MyDrive/natural-presenter-ai/vn_01_f5_dataset.zip`
3. Run the cells below in order. Checkpoints are **automatically saved directly to Google Drive** every 500 updates so you never lose progress!
4. After listening in Cell 6, download the best `.pt` / `.safetensors` checkpoint to your laptop at:
   `D:\natural-presenter-ai\profiles\<speaker_id>\lora\`

In [ ]:
#@title 1. Configuration, Mount Google Drive & Verify GPU
import os, torch
from pathlib import Path
from google.colab import drive

SPEAKER_ID = "vn_01"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/natural-presenter-ai"  #@param {type:"string"}
EPOCHS = 80  #@param {type:"integer"}
LEARNING_RATE = 1e-5  #@param {type:"number"}
BATCH_FRAMES = 3200  #@param {type:"integer"}
WARMUP_UPDATES = 100  #@param {type:"integer"}
SAVE_PER_UPDATES = 500  #@param {type:"integer"}

drive.mount("/content/drive")

ZIP_PATH = Path(DRIVE_ROOT) / f"{SPEAKER_ID}_f5_dataset.zip"
DRIVE_CKPT_DIR = Path(DRIVE_ROOT) / "ckpts" / SPEAKER_ID
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"Dataset zip not found at {ZIP_PATH}. Upload it to Google Drive first!"
print(f"Found dataset zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Checkpoints will auto-save to: {DRIVE_CKPT_DIR}")
print(f"GPU Available: {torch.cuda.is_available()} | Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
#@title 2. Clone & Install F5-TTS (Editable Repo for Standard data/ and ckpts/ Paths)
import os

if not os.path.exists("/content/F5-TTS"):
    !git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS

!pip install -q -e /content/F5-TTS bitsandbytes accelerate datasets soundfile
print("F5-TTS installed cleanly!")

In [ ]:
#@title 3. Extract Dataset, Build F5-TTS Arrow Format & Link Checkpoints to Google Drive
import csv, json, os, shutil, zipfile
from pathlib import Path
import soundfile as sf
from datasets.arrow_writer import ArrowWriter

RAW_DIR = Path(f"/content/datasets/{SPEAKER_ID}")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_DIR)

# Build F5-TTS data directory: /content/F5-TTS/data/{SPEAKER_ID}_pinyin
# Note: F5TTS_v1_Base uses the 2545-token pretrained vocab (English letters remain raw chars,
# zero G2P phoneme normalization, preserving non-native accents while matching pretrained weights).
F5_DATA_DIR = Path(f"/content/F5-TTS/data/{SPEAKER_ID}_pinyin")
F5_DATA_DIR.mkdir(parents=True, exist_ok=True)

rows = []
durations = []
with (RAW_DIR / "metadata.csv").open("r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")
    header = next(reader, None)
    for rel_wav, text in reader:
        abs_wav = (RAW_DIR / rel_wav.strip()).resolve()
        if not abs_wav.exists():
            continue
        dur = float(sf.info(str(abs_wav)).duration)
        rows.append({"audio_path": str(abs_wav), "text": text.strip(), "duration": dur})
        durations.append(dur)

assert len(rows) > 0, "No valid WAV rows found!"

with ArrowWriter(path=str(F5_DATA_DIR / "raw.arrow")) as writer:
    for r in rows:
        writer.write(r)
    writer.finalize()

(F5_DATA_DIR / "duration.json").write_text(json.dumps({"duration": durations}, indent=2), encoding="utf-8")

# Copy exact pretrained vocab.txt matching F5TTS_v1_Base (vocab_size = 2545)
pretrained_vocab = Path("/content/F5-TTS/src/f5_tts/infer/examples/vocab.txt")
shutil.copy2(pretrained_vocab, F5_DATA_DIR / "vocab.txt")

# Symlink /content/F5-TTS/ckpts/{SPEAKER_ID} -> Google Drive so every checkpoint persists automatically
F5_CKPTS_ROOT = Path("/content/F5-TTS/ckpts")
F5_CKPTS_ROOT.mkdir(parents=True, exist_ok=True)
LINK_PATH = F5_CKPTS_ROOT / SPEAKER_ID
if LINK_PATH.is_symlink() or LINK_PATH.exists():
    if LINK_PATH.is_symlink():
        LINK_PATH.unlink()
    else:
        shutil.rmtree(LINK_PATH)
os.symlink(str(DRIVE_CKPT_DIR), str(LINK_PATH))

print(f"Dataset ready: {len(rows)} clips | {sum(durations) / 60:.2f} minutes")
print(f"Arrow path:    {F5_DATA_DIR / 'raw.arrow'}")
print(f"Vocab lines:   {len((F5_DATA_DIR / 'vocab.txt').read_text(encoding='utf-8').splitlines())}")
print(f"Checkpoint symlink: {LINK_PATH} -> {DRIVE_CKPT_DIR}")

In [ ]:
#@title 4. Launch F5-TTS Fine-Tuning on Tesla T4 (15GB VRAM — fp16 + 8-bit AdamW)
from pathlib import Path

# Giảm DataLoader workers xuống 2 cho an toàn với RAM/CPU của T4 Colab
trainer_py = Path("/content/F5-TTS/src/f5_tts/model/trainer.py")
src = trainer_py.read_text(encoding="utf-8")
src = src.replace("num_workers=16", "num_workers=2").replace("num_workers=4", "num_workers=2")
trainer_py.write_text(src, encoding="utf-8")

!accelerate launch --mixed_precision=fp16 /content/F5-TTS/src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_v1_Base \
    --dataset_name {SPEAKER_ID} \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 1800 \
    --batch_size_type frame \
    --grad_accumulation_steps 2 \
    --epochs 60 \
    --num_warmup_updates 100 \
    --save_per_updates 400 \
    --last_per_updates 200 \
    --keep_last_n_checkpoints 4 \
    --finetune \
    --tokenizer pinyin \
    --bnb_optimizer

In [ ]:
#@title 5. Listen & Compare: Zero-Shot Base vs. Your Fine-Tuned Checkpoints!
import sys
from pathlib import Path
from IPython.display import Audio, display

# Đảm bảo Python nhận diện được thư viện F5-TTS trong thư mục cài đặt
if "/content/F5-TTS/src" not in sys.path:
    sys.path.append("/content/F5-TTS/src")

from f5_tts.api import F5TTS

TEST_SCRIPT = (
    "Hello everyone, today I want to, uh, walk you through our experiment results. "
    "We tested two model architectures across eighty trials, and, like, we noticed a clear drop "
    "in baseline error after tuning the encoder resolution."
)

REF_WAV = str(RAW_DIR / "refs/presentation_ref.wav")
REF_TXT = (RAW_DIR / "refs/presentation_ref.txt").read_text(encoding="utf-8").strip()

print("=== 1. Base F5TTS_v1_Base (Zero-Shot Baseline) ===")
base_tts = F5TTS(device="cuda")
wav_base, sr, _ = base_tts.infer(ref_file=REF_WAV, ref_text=REF_TXT, gen_text=TEST_SCRIPT, nfe_step=32)
display(Audio(wav_base, rate=sr))

ckpts = sorted(
    [p for p in DRIVE_CKPT_DIR.glob("model_*.pt") if not p.name.startswith("pretrained_")]
    + [p for p in DRIVE_CKPT_DIR.glob("model_*.safetensors") if not p.name.startswith("pretrained_")]
)
print(f"\nFound {len(ckpts)} fine-tuned checkpoint(s) in {DRIVE_CKPT_DIR}:")
for ckpt in ckpts:
    print(f"\n=== Fine-Tuned: {ckpt.name} ({ckpt.stat().st_size / 1e6:.1f} MB) ===")
    ft_tts = F5TTS(ckpt_file=str(ckpt), device="cuda")
    wav_ft, sr, _ = ft_tts.infer(ref_file=REF_WAV, ref_text=REF_TXT, gen_text=TEST_SCRIPT, nfe_step=32)
    display(Audio(wav_ft, rate=sr))

print(
    "\nPick the checkpoint that sounds most authentically like you (accent + natural 'uh'), "
    "download it from Google Drive, and place it into:\n"
    f"  D:\\natural-presenter-ai\\profiles\\{SPEAKER_ID}\\lora\\"
)

In [ ]:
#@title 6. Export Slim FP16 Inference Model (~643 MB instead of 3.4 GB)
import torch
from safetensors.torch import save_file

BEST_CKPT = "model_4000.pt"  #@param {type:"string"}
src_ckpt = DRIVE_CKPT_DIR / BEST_CKPT
ckpt = torch.load(str(src_ckpt), map_location="cpu", weights_only=True)
ema = {k: v.half() for k, v in ckpt["ema_model_state_dict"].items() if isinstance(v, torch.Tensor)}
out_path = DRIVE_CKPT_DIR / f"{SPEAKER_ID}_{BEST_CKPT.replace('.pt', '_fp16.safetensors')}"
save_file(ema, str(out_path))
print(f"Saved slim inference checkpoint: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
